# Solution: Day 1 + Day 2 Homework

This notebook covers:
1. **Day 1** - Web page summarizer using OpenAI (snarky personality)
2. **Day 1 Exercise** - Email subject line suggester (commercial use case)
3. **Day 2 Homework** - Web page summarizer upgraded to use Ollama (local, free, open-source)

---
# Part 1 — Day 1: Web Summarizer with OpenAI

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key found. Add OPENAI_API_KEY to your .env file.")
else:
    print("API key loaded successfully.")

openai = OpenAI()

In [ ]:
# System prompt: snarky summarizer personality
system_prompt = """
You are a snarky assistant that analyzes the contents of a website
and provides a short, snarky, humorous summary, ignoring text that
might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block.
"""

# User prompt prefix
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, summarize these too.

"""

def messages_for(website_text):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website_text}
    ]

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages_for(website)
    )
    return response.choices[0].message.content

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [ ]:
# Try it out
display_summary("https://edwarddonner.com")

In [ ]:
display_summary("https://anthropic.com")

---
# Part 2 — Day 1 Exercise: Email Subject Line Suggester

A practical commercial use case: paste an email body and get a concise, professional subject line suggestion.
This is the kind of feature that could be embedded directly into email clients like Outlook or Gmail.

In [ ]:
# System prompt for subject line generation
email_system_prompt = """
You are a professional email assistant.
When given the body of an email, you suggest ONE concise, clear, and professional
subject line for it. The subject line should be no longer than 10 words.
Respond with just the subject line — no explanation, no prefix like 'Subject:'.
"""

def suggest_subject_line(email_body: str) -> str:
    messages = [
        {"role": "system", "content": email_system_prompt},
        {"role": "user", "content": f"Email body:\n\n{email_body}"}
    ]
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )
    return response.choices[0].message.content.strip()

In [ ]:
# Example 1: Project update email
email_1 = """
Hi Team,

I wanted to give everyone a quick update on the Q3 product launch.
We have completed the backend integration and QA testing is now underway.
The expected release date is still on track for September 15th.
Please review the attached timeline and flag any blockers by Friday.

Thanks,
Imran
"""

subject = suggest_subject_line(email_1)
print(f"Suggested subject: {subject}")

In [ ]:
# Example 2: Meeting request email
email_2 = """
Dear Sarah,

I hope you're doing well. I'd love to schedule a 30-minute call to discuss
the potential partnership between our companies. I'm available Monday or
Tuesday afternoon next week — whatever works best for you.

Looking forward to connecting!
Best,
Imran
"""

subject = suggest_subject_line(email_2)
print(f"Suggested subject: {subject}")

In [ ]:
# Example 3: Customer support email
email_3 = """
Hello Support Team,

I have been experiencing issues with logging into my account for the past two days.
I've tried resetting my password but I'm not receiving the reset email.
My account email is user@example.com. Please help resolve this as soon as possible
as I need access for an important deadline tomorrow.

Regards,
Imran Ali
"""

subject = suggest_subject_line(email_3)
print(f"Suggested subject: {subject}")

---
# Part 3 — Day 2 Homework: Web Summarizer with Ollama

Upgraded version of the Day 1 summarizer — now using a **free, local, open-source model** via Ollama.

**Benefits over OpenAI:**
- No API costs
- Data never leaves your machine (privacy)
- Works offline

**Prerequisites:**
1. Install Ollama from [ollama.com](https://ollama.com)
2. Run `ollama pull llama3.2` in a terminal
3. Ensure Ollama is running (`ollama serve` if needed)

In [ ]:
import requests as http_requests

# Check Ollama is running
try:
    result = http_requests.get("http://localhost:11434")
    print(result.content.decode())
except Exception as e:
    print(f"Ollama is not running. Please run 'ollama serve' in a terminal. Error: {e}")

In [ ]:
# Pull the model (only needed once)
# Use llama3.2:1b instead if your machine is lower-powered
!ollama pull llama3.2

In [ ]:
# Ollama exposes an OpenAI-compatible endpoint locally
# We simply point the OpenAI client at it

OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_MODEL = "llama3.2"  # change to "llama3.2:1b" for smaller machines

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# Reuse the same system prompt from Day 1
ollama_system_prompt = """
You are a snarky assistant that analyzes the contents of a website
and provides a short, snarky, humorous summary, ignoring text that
might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block.
"""

ollama_user_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, summarize these too.

"""

def ollama_messages_for(website_text):
    return [
        {"role": "system", "content": ollama_system_prompt},
        {"role": "user", "content": ollama_user_prefix + website_text}
    ]

def summarize_with_ollama(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=ollama_messages_for(website)
    )
    return response.choices[0].message.content

def display_summary_ollama(url):
    print(f"Summarizing with Ollama ({OLLAMA_MODEL}): {url}\n")
    summary = summarize_with_ollama(url)
    display(Markdown(summary))

In [ ]:
# Try the Ollama summarizer
display_summary_ollama("https://edwarddonner.com")

In [ ]:
display_summary_ollama("https://anthropic.com")

---
## Side-by-side Comparison: OpenAI vs Ollama

Run both models on the same URL and compare the output quality and style.

In [ ]:
def compare_models(url):
    print(f"URL: {url}\n")
    print("=" * 60)
    print("OpenAI (gpt-4.1-mini):")
    print("=" * 60)
    display(Markdown(summarize(url)))

    print("=" * 60)
    print(f"Ollama ({OLLAMA_MODEL}):")
    print("=" * 60)
    display(Markdown(summarize_with_ollama(url)))

compare_models("https://edwarddonner.com")